<a href="https://colab.research.google.com/github/rene-aum/Hermes/blob/Moises/Asignacion/nb3_asignacion_edas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automatización de Asignación de leads de crédito (Apis & Contingencia)
Recuerda que para este punto ya debiste subir al menos la **base de clientes con Corte 1** del día.
Además, por favor, **asegúrate de colocar correctamente**
1. La cosecha que estás asignando.
2. Si vas a escribir automáticamente en la torre de control (versión 2)

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

id_drive_edas = '14rh8YbJUtXiyfNdSmUWueHRyoet-SezJ'
id_drive_hist = '1zvW-Dxow9gz1Dnbpg_jO7my4wadDvJDW'
id_drive_catalogos = '1xQkepXoIoEHNgdYUaLAT7FtDnuzYo1qb'
id_drive_salidas = '1qZlTS_buGs876ojYKuBmOJ3n5zzWLmBS'
id_sheets_tc2 = '1k8rguLeF1O33XCaVDxPiQ1C4SbxLDSIeqNcriYtsF-k'

fh_salida =  datetime.now(ZoneInfo("America/Mexico_City")).strftime('%Y-%m-%d')   # '2026-02-17'
fh_salida_dt = datetime.strptime(fh_salida, '%Y-%m-%d')
dia_salida = str(fh_salida_dt.day).zfill(2)
mes_salida = str(fh_salida_dt.month).zfill(2)
anio_salida = str(fh_salida_dt.year).zfill(4)
fh_de_asignacion = fh_salida_dt.strftime('%d-%m-%Y')

cosecha = 'Cosecha Mar 26' #@param{type:'string'}

nb_ultimo_corte = 'ultimoCorte_edas.csv'
nb_carpeta_ctes_mes = f'{anio_salida}{mes_salida}'
nb_ctes_csv = f'report_{anio_salida}{mes_salida}{dia_salida}'
nb_sheet_salida = f'Salidas {fh_salida}'

dicc_espacios = {'Reforma 510':'torre','MetrÃ³poli Patriotismo':'patriotismo','Samara SatÃ©lite':'samara'}
dicc_espacios2 = {'MetrÃ³poli Patriotismo': 'Metrópoli Patriotismo','Samara SatÃ©lite': 'Samara Satélite'}
dicc_espacios3 = {'torre':'Reforma 510','patriotismo':'Metrópoli Patriotismo','samara':'Samara Satélite'}

actualizar_tc = 'S' #@param{type:'string'}['S','N']
validar_montos = 'N' #@param{type:'string'}['S','N']

In [ ]:
dia_salida, mes_salida, anio_salida, nb_carpeta_ctes_mes, nb_ctes_csv, fh_de_asignacion

('06', '03', '2026', '202603', 'report_20260306', '06-03-2026')

## Paqueterías

In [ ]:
import os

from_drive = True  # same flag you use everywhere

if os.environ.get("HERMES_BOOTSTRAPPED") != "1":
    # ---------- GIT ON COLAB ONLY ----------
    try:
        from google.colab import userdata

        git_token = userdata.get('gitToken')
        git_user = userdata.get('gitUser')
        git_url = f'https://{git_token}@github.com/rene-aum/Hermes.git'
        branch_to_pull = 'dev'

        os.chdir('/content')

        if not os.path.isdir('Hermes'):
            !git clone {git_url}

        %cd Hermes
        !git fetch origin {branch_to_pull}
        !git checkout {branch_to_pull}
        !git pull origin {branch_to_pull}

        !pip install -r utils/src/requirements.txt
        %cd Asignacion

    except Exception as e:
        print(e)
        print('Running in other environment not colab probably!')

    # ---------- DRIVE + SHEETS ----------
    if from_drive:
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        import gspread
        from google.auth import default
        from gspread_dataframe import set_with_dataframe
        import gdown

        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)

        creds, _ = default()
        gc = gspread.authorize(creds)

    os.environ["HERMES_BOOTSTRAPPED"] = "1"
else:
    print("Bootstrap already done, assuming orchestrator ran it.")

Cloning into 'Hermes'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 158 (delta 91), reused 52 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 102.29 KiB | 1.11 MiB/s, done.
Resolving deltas: 100% (91/91), done.
/content/Hermes
From https://github.com/rene-aum/Hermes
 * branch            dev        -> FETCH_HEAD
Branch 'dev' set up to track remote branch 'dev' from 'origin'.
Switched to a new branch 'dev'
From https://github.com/rene-aum/Hermes
 * branch            dev        -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.1 MB/s eta 0:00:00
/content/Hermes/Asignacion


In [ ]:
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
import sys
sys.path.append('..')
sys.path.append('../..')
from utils.utils import (get_dates_dataframe,
                       add_year_week,
                       custom_read,
                       process_columns,
                       remove_accents)

from utils.drive_toolbox import(from_drive_to_local,
                             get_last_modification_date_drive,
                             create_sheets_in_drive_folder,
                             update_sheets_in_drive_folder,
                             read_from_google_sheets,
                             list_file_ids_for_drive_folder,
                             create_csv_file_in_drive_folder,
                             write_csv_to_drive,
                             read_csv_from_drive,
                             append_dataframe_to_google_sheet_from_range,
                             send_google_chat_notification)
from utils.src.constants import (atlas_consumo_output_folder_id,
                           consumo_sheets_ids_dict,
                           folder_id_bauto_gabo,
                           id_reporte_ventas,
                           id_edas_referenciados,
                           id_torre_de_control
                           )


warnings.filterwarnings('ignore')



In [ ]:
# EXTRAS

import math
from zoneinfo import ZoneInfo
import re

# --------- BUSCAR EN SUBCARPETAS -------------------------------------------
from googleapiclient.discovery import build
creds, _ = default()

servicedrive = build("drive", "v3", credentials=creds)
service_sheets = build("sheets", "v4", credentials=creds)

FOLDER_MIME = "application/vnd.google-apps.folder"

def listar_archivos(folder_id, mime_types=None):
    """
    folder_id: ID de la carpeta raíz
    mime_types: None | string | lista de strings
    """
    if isinstance(mime_types, str):
        mime_types = [mime_types]

    resultados = {}

    def recorrer(fid):
        page_token = None
        while True:
            resp = servicedrive.files().list(
                q=f"'{fid}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType)",
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()

            for f in resp.get("files", []):
                if f["mimeType"] == FOLDER_MIME:
                    recorrer(f["id"])
                else:
                    if mime_types is None or f["mimeType"] in mime_types:
                        resultados[f["name"]] = f["id"]

            page_token = resp.get("nextPageToken")
            if not page_token:
                break

    recorrer(folder_id)
    return resultados


# -------------- LEER CON ENCODING ------------------------------------------
import io
import pandas as pd
from googleapiclient.http import MediaIoBaseDownload

def read_csv_from_drive_v3(drive_service, file_id, **read_csv_kwargs):
    request = drive_service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)

    done = False
    while not done:
        status, done = downloader.next_chunk()

    fh.seek(0)
    return pd.read_csv(fh, **read_csv_kwargs)


# import io
# def read_csv_from_drive2(drive, file_id, encoding="latin-1", **read_csv_kwargs):
#     f = drive.CreateFile({"id": file_id})
#     f.FetchContent()
#     b = f.content.getvalue()  # bytes
#     return pd.read_csv(io.BytesIO(b), encoding=encoding, **read_csv_kwargs)

# ------------- Todas las columnas ------------------------------------------
pd.set_option('display.max_columns', 100)

# ------------- IMPRIMIR CON COLORES ----------------------------------------
class color:
   PURPLE = '\033[95m'
   CYAN = '\033[94m'
   DARKCYAN = '\033[36m'
   BLUE = '\033[94m'
   GREEN = '\033[92m'
   YELLOW = '\033[93m'
   RED = '\033[91m'
   BOLD = '\033[1m'
   UNDERLINE = '\033[4m'
   END = '\033[0m'

def borrar_hojas(spreadsheet_id, nb_hojas):
    spreadsheet = service_sheets.spreadsheets().get(
        spreadsheetId=spreadsheet_id
    ).execute()

    sheet_ids = []
    for sheet in spreadsheet["sheets"]:
        if sheet["properties"]["title"] in nb_hojas:
            sheet_ids.append(sheet["properties"]["sheetId"])

    request = {
        "requests": [
            {"deleteSheet": {"sheetId": s_id}}
            for s_id in sheet_ids
            ]
    }

    service_sheets.spreadsheets().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body=request
    ).execute()
    print(f"{nb_hojas} eliminada(s)")
    return

    print("Hoja no encontrada")

def crear_hojas_sheets(spreadsheet_id, nb_hojas, quitar_cuadricula = True, fila_congelada = 1):
  """
  nb_hojas: lista con los nombres de las hojas a crear
  quitar_cuadricula: True | False es para hacer invisibles los bordes/cuadrícula de las celdas
  """

  request = {
      "requests": [
          {"addSheet": {"properties": {"title": nombre,
                                       'gridProperties': {'hideGridlines':quitar_cuadricula,
                                                          'frozenRowCount':fila_congelada}
                                       }
                        }
           }
          for nombre in nb_hojas
      ]
  }

  service_sheets.spreadsheets().batchUpdate(
      spreadsheetId = spreadsheet_id,
      body=request
  ).execute()
  print(f'{nb_hojas} creadas')


def formato_hojas_sheets(sheets_id, nb_hojas, n_columnas, tamanio_letra = 11, letra = 'Source Serif 4', rgb_encabezado = [0.1, 0.3, 0.7], ):
  spreadsheet = service_sheets.spreadsheets().get(
      spreadsheetId=sheets_id
  ).execute()

  sheet_ids = []
  for sheet in spreadsheet["sheets"]:
      if sheet["properties"]["title"] in nb_hojas:
          sheet_ids.append(sheet["properties"]["sheetId"])

  # requests de formato
  requests = []
  for sh_id in sheet_ids:
    # A) Fuente para TODA la hoja
    r_global = {
        "repeatCell": {
            "range": {
                "sheetId": sh_id
            },
            "cell": {
                "userEnteredFormat": {
                    "textFormat": {
                        "fontFamily": letra,
                        "fontSize": tamanio_letra
                    }
                }
            },
            "fields": "userEnteredFormat.textFormat(fontFamily,fontSize)"
        }
    },

    # B) Color solo en encabezado (fila 1)
    r_encabezado = {
        "repeatCell": {
            "range": {
                "sheetId": sh_id,
                "startRowIndex": 0,
                "endRowIndex": 1,
                'startColumnIndex':0,
                'endColumnIndex':n_columnas
            },
            "cell": {
                "userEnteredFormat": {
                    "backgroundColor": {
                        "red": rgb_encabezado[0],
                        "green": rgb_encabezado[1],
                        "blue": rgb_encabezado[2]
                    },
                    "textFormat": {
                        "bold": True,
                        "foregroundColor": {
                            "red": 1,
                            "green": 1,
                            "blue": 1
                        }
                    }
                }
            },
            "fields": "userEnteredFormat(backgroundColor,textFormat.bold,textFormat.foregroundColor)"
        }
    }
    r_anchoColumnas = {
        "autoResizeDimensions": {
            "dimensions": {
                "sheetId": sh_id,
                "dimension": "COLUMNS",
                "startIndex": 0,
                "endIndex": 20   # ajusta primeras 20 columnas
            }
        }
    }
    requests.append(r_global)
    requests.append(r_encabezado)
    requests.append(r_anchoColumnas)

  service_sheets.spreadsheets().batchUpdate(
      spreadsheetId=sheets_id,
      body={"requests": requests}
      ).execute()

## Leemos base de solicitudes y nos quedamos con los nuevos desde el último corte

In [ ]:
fh_corte = datetime.now(ZoneInfo("America/Mexico_City")).strftime('%d-%m-%Y %H')
insumos_edas = list_file_ids_for_drive_folder(drive, id_drive_edas)
edas_full = read_from_google_sheets(gc, insumos_edas['EdasImport'], 'Hoja 1')
edas_full['Folio Preautorizado'] = pd.to_numeric(edas_full['Folio Preautorizado'].astype(str).str.strip(), errors = 'coerce').astype('Int64')
edas_full['Teléfono celular del cliente'] = pd.to_numeric(edas_full['Teléfono celular del cliente'].replace(r'[-\s]', '', regex=True), errors = 'coerce').astype('Int64')
edas_full = edas_full[(edas_full['Folio Preautorizado'].notna()) & (edas_full['Teléfono celular del cliente'].notna())]
edas_full['fh_corte'] = fh_corte

edas_ultimoCorte = read_csv_from_drive(drive, insumos_edas[nb_ultimo_corte])
edas_ultimoCorte['Folio Preautorizado'] = pd.to_numeric(edas_ultimoCorte['Folio Preautorizado'].astype(str).str.strip(), errors = 'coerce').astype('Int64')
edas_ultimoCorte['Teléfono celular del cliente'] = pd.to_numeric(edas_ultimoCorte['Teléfono celular del cliente'].replace(r'[-\s]', '', regex=True), errors = 'coerce').astype('Int64')
edas_ultimoCorte = edas_ultimoCorte[(edas_ultimoCorte['Folio Preautorizado'].notna()) & (edas_ultimoCorte['Teléfono celular del cliente'].notna())]

In [ ]:
edas_ult = edas_full.copy()
edas_penult = edas_ultimoCorte.copy()

cols_edas = {'Folio Preautorizado':'folio','Nombre de Cliente':'nb_comprador','Espacio':'espacio', 'Observaciones de contacto':'obs_contacto','Teléfono celular del cliente':'phone'
          ,'Fecha nacimiento folio ':'fh_creacion_folio','Medio de contacto preferencia del cliente':'pref_contacto'}

edas_ult.rename(columns = cols_edas, inplace=True)
edas_ult['fh_corte'] = pd.to_datetime(edas_ult['fh_corte'],format='%d-%m-%Y %H')

edas_penult.rename(columns = cols_edas, inplace=True)
edas_penult['fh_corte'] = pd.to_datetime(edas_penult['fh_corte'],format='%d-%m-%Y %H')

sols_edas = edas_ult[~edas_ult['folio'].isin(edas_penult['folio'].unique())].copy().reset_index(drop=True) # Hacemos la diferencia vs el último corte
sols_edas['tp_solicitud'] = 'EDA Crédito'

phone_regex = re.compile(r'([0-9\s\+-]{8,20})')
sols_edas['phone2'] = sols_edas['obs_contacto'].str.strip().str.replace(r'[()-]','', regex = True).str.extract(phone_regex)
sols_edas['phone2'] = sols_edas['phone2'].str.replace(r'\D','', regex=True) # Quitamos todo lo que NO sea número
sols_edas['phone'] = np.where(sols_edas['phone'].notna(), sols_edas['phone'], sols_edas['phone2'])
sols_edas = sols_edas[sols_edas['phone'].notna()]

print(f'{color.BOLD}{color.CYAN}Tenemos {sols_edas.shape[0]} solicitudes nuevas{color.END}')

if validar_montos == 'S':
  sols_edas['Monto del certificado '] = pd.to_numeric(sols_edas['Monto del certificado '].replace(',','',regex=True), errors='coerce').astype(float)
  sols_edas = sols_edas[sols_edas['Monto del certificado '] >= 99]
  print(f'{color.BOLD}{color.CYAN}De las cuales {sols_edas.shape[0]} sabemos que son por montos mayores a 100k{color.END}')

display(sols_edas)
pdds_sf1 = sols_edas.copy()

Tenemos 10 solicitudes nuevas


,Marca temporal,Selecciona tu puesto,Nombre completo del Empleado,Usuario M,folio,nb_comprador,obs_contacto,espacio,(En caso de ECA OPCIONAL)\nNombre del Banquero Asociado al Folio,phone,Monto del certificado,fh_creacion_folio,pref_contacto,Días antig. folio a fecha asignación,Día máximo,Número del crédito formalizado y desembolsado,Fecha de desembolso,Comentarios del espacio,Formalizo Automarket,Columna 1,Unnamed: 21,fh_corte,tp_solicitud,phone2
0,5/03/2026 14:14:28,ATENTO,NaN,NaN,9733695,KIMBERLY AREYSI MIGUEL ARGUETA,WHATSAPP: 5578288005,NaN,NaN,9994150476,200000.0,05/03/2026,Whatsapp,0.593377,15/03/2026 14:14:28,NaN,NaN,NaN,NaN,NaN,5/03/2026,2026-03-06 18:00:00,EDA Crédito,5578288005
1,5/03/2026 15:56:17,ATENTO,NaN,NaN,9720835,LUIS REBOLLO TINOCO,WHATSAPP 5621830204,NaN,NaN,5621830204,115900.0,03/03/2026,Whatsapp,2.664091,15/03/2026 15:56:17,NaN,NaN,NaN,NaN,NaN,5/03/2026,2026-03-06 18:00:00,EDA Crédito,5621830204
2,5/03/2026 15:59:51,ATENTO,NaN,NaN,9724896,JUAN CRUZ SANCHEZ,5628470884 WHATS APP,NaN,NaN,5628470884,150000.0,03/03/2026,Whatsapp,2.666567,15/03/2026 15:59:51,NaN,NaN,NaN,NaN,NaN,5/03/2026,2026-03-06 18:00:00,EDA Crédito,5628470884
3,5/03/2026 17:32:19,ATENTO,NaN,NaN,9724817,JOSE ANTONIO RODRIGUEZ SANTIAGO,WHATSAPP: 5571295559,NaN,MA,5571295559,218700.0,03/03/2026,Whatsapp,2.730775,15/03/2026 17:32:19,NaN,NaN,NaN,NaN,NaN,5/03/2026,2026-03-06 18:00:00,EDA Crédito,5571295559
4,5/03/2026 17:33:30,ATENTO,NaN,NaN,9732002,BLANCA ESTELA BAUTISTA MEJIA,LLAMADA: 5649965974,NaN,NaN,5649965974,151600.0,04/03/2026,Llamada telefónica,1.7316,15/03/2026 17:33:30,NaN,NaN,NaN,NaN,NaN,5/03/2026,2026-03-06 18:00:00,EDA Crédito,5649965974
5,6/03/2026 9:19:08,ATENTO,NaN,NaN,9721172,SAMUEL HERMENEGILDO SANCHEZ,WHATSAPP 7121598144,NaN,NaN,7121598144,150000.0,03/03/2026,Whatsapp,3.388288,16/03/2026 9:19:08,NaN,NaN,NaN,NaN,NaN,6/03/2026,2026-03-06 18:00:00,EDA Crédito,7121598144
6,6/03/2026 11:38:54,ATENTO,NaN,NaN,9722596,JESUS HERNANDEZ CRUZ,WHATSAPP TEL:7291650758,NaN,NaN,7291650758,115000.0,03/03/2026,Whatsapp,3.485348,16/03/2026 11:38:54,NaN,NaN,NaN,NaN,NaN,6/03/2026,2026-03-06 18:00:00,EDA Crédito,7291650758
7,6/03/2026 13:23:30,ATENTO,NaN,NaN,9723697,SWEETY MELISSA CORONA MADRIGAL,WHATSAPP 5573381915,NaN,NaN,5573381915,200000.0,03/03/2026,Whatsapp,3.557987,16/03/2026 13:23:30,NaN,NaN,NaN,NaN,NaN,6/03/2026,2026-03-06 18:00:00,EDA Crédito,5573381915
8,6/03/2026 14:41:49,ATENTO,NaN,NaN,9554713,AARON AMAURY SANTANA SEGURA,WHATSAPP: 3318309768,NaN,NaN,5611837772,150000.00,03/02/2026,Whatsapp,31.612372,16/03/2026 14:41:49,NaN,NaN,NaN,NaN,NaN,6/03/2026,2026-03-06 18:00:00,EDA Crédito,3318309768
9,6/03/2026 15:34:47,ATENTO,NaN,NaN,9581098,AGRIPINO ESPINOZA VAZQUEZ,WHATSAPP TELEFONO: 5531409786,NaN,NaN,5531409786,150000.00,08/02/2026,Whatsapp,26.649155,16/03/2026 15:34:47,NaN,NaN,NaN,NaN,NaN,6/03/2026,2026-03-06 18:00:00,EDA Crédito,5531409786


In [ ]:
# CELDA DE VALIDACIÓN
fh_final_sols, fh_inicial_sols = edas_ult['fh_corte'].max().strftime('%d/%m/%Y'), edas_penult['fh_corte'].max().strftime('%d/%m/%Y')
fhs_creacion_ls = pd.to_datetime(sols_edas['Marca temporal']).dt.strftime('%d/%m/%Y').unique().tolist()

if not all([x <= fh_final_sols and x >= fh_inicial_sols for x in fhs_creacion_ls if x!=np.nan]):
    print(f"{color.RED}Hay solicitudes con fechas fuera del rango de cortes de actualización. Probablemente agregaron folio y/o celular a una solicitud anterior{color.END}")
    # raise SystemExit

Hay solicitudes con fechas fuera del rango de cortes de actualización. Probablemente agregaron folio y/o celular a una solicitud anterior


## Buscamos datos de solicitudes nuevas en histórico y nos quedamos con las que cumplan definición de leads nuevos

In [ ]:
# Leemos y ordenamos histórico, corregimos nombre de asesora y generamos variable de último lead
id_hist = '1zvW-Dxow9gz1Dnbpg_jO7my4wadDvJDW'
nb_hist = '_latest.csv'
hist = list_file_ids_for_drive_folder(drive, id_hist)
hist = [v for i,v in hist.items() if nb_hist in i ][0]
hist = read_csv_from_drive_v3(servicedrive, hist, encoding='latin-1' )

# Primero los registros más recientes
hist['fecha de asignacion'] = pd.to_datetime(hist['fecha de asignacion'], format = '%Y-%m-%d')
hist = hist.sort_values(by='fecha de asignacion', ascending=False)
#------------------------------------

hist['asesor espacio'] = hist['asesor espacio'].str.replace('SaldaÃ±a','Saldaña')
hist['conteo_leads'] = hist['id lead'].str.replace('|','0').str[-6:].astype(int)
ultimo_lead = hist['conteo_leads'].max()
print(f'{color.BLUE}El último lead, a partir del cual vamos a empezar a asignar en este proceso es el {ultimo_lead}{color.END}')
hist.sample(1)

El último lead, a partir del cual vamos a empezar a asignar en este proceso es el 20537


,id lead,origen automarket,cosecha,id comprador,folio bauto tc,nombre comprador,mail comprador,telefono comprador,asesor credito,espacio automarket,asesor espacio,fecha de asignacion,estatus de lead,fecha_de_proceso,flag_torre_v2,flag salio de cerrado,fecha de reactivacion credito,fecha de reactivacion eam,conteo_leads
20493,LAA-020176,Apartado,Cosecha Mar 26,633589,x,Miguel Merino,mimerino@mac.com,5535661570,Jose Luis Acevedo,Reforma 510,Ana Karen Castro,2026-03-04,CERRADO,2026-03-06 15:36:27,1.0,NaN,NaN,NaN,20176


In [ ]:

hist_fon = hist[['telefono comprador','id lead','estatus de lead']].copy().drop_duplicates('telefono comprador').dropna(subset='telefono comprador')

hist_fon.columns = ['phone','id_lead','estatus_lead']

pdds_sf1['fon'] = pd.to_numeric(pdds_sf1['phone'].astype(str).str.strip().str[-10:], errors = 'coerce').astype('Int64')
hist_fon['phone'] = pd.to_numeric(hist_fon['phone'].astype(str).str.strip().str[-10:], errors = 'coerce').astype('Int64')

pdds_sf2 = pdds_sf1.merge(hist_fon[['phone','id_lead','estatus_lead']], how = 'left', on = 'phone', suffixes = ['','_confon'])

cerrados = ['COMPRA EXITOSA ','COMPRA EXITOSA','CERRADO','NA']

nvos_leads = ( (pdds_sf2['id_lead'].isna()) | (pdds_sf2['estatus_lead'].isin(cerrados) ) )

leads_ok = pdds_sf2[~nvos_leads].copy()
leads_nvos = pdds_sf2[nvos_leads].copy()

print(len(leads_ok), len(leads_nvos), len(pdds_sf2))

if not len(leads_ok) + len(leads_nvos) == len(pdds_sf2):
  print(f"{color.BOLD}{color.CYAN}La clasificación de leads ok (pedidos que no requieren lead nuevo) y leads nuevos está perdiendo alguno(s) de los pedidos con los que iniciamos {color.END}")
  raise SystemExit

0 10 10


In [ ]:
leads_nvos['id_comprador'] = 'x'

email_regex = re.compile(r'([A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})')
leads_nvos['email'] = leads_nvos['obs_contacto'].astype(str).str.lower().str.extract(email_regex)
leads_nvos['email'] = np.where(leads_nvos['email'].isna(),'x',leads_nvos['email'])

leads_nvos = leads_nvos[['id_comprador','phone','email','nb_comprador','tp_solicitud','folio']].drop_duplicates(['phone'], keep='first').reset_index(drop=True)

print(f'{color.BOLD}{color.CYAN} Tenemos {len(leads_nvos)} leads nuevos.{color.END}')

 Tenemos 10 leads nuevos.


## Asignaciones

### Asignación espacio

In [ ]:
# Primero asignamos espacio

hist_credito_activos = hist.copy()
hist_credito_activos = hist_credito_activos[hist_credito_activos['origen automarket'].str.contains(r'Contingencia|Apificado|API') &
                                             ((~hist_credito_activos['estatus de lead'].isin(cerrados)) & (hist_credito_activos['espacio automarket']!='PRUEBA'))]

leads_credito_activos = hist_credito_activos.groupby(['espacio automarket']).agg({'id lead':'nunique'}).reset_index()
leads_credito_activos = leads_credito_activos.sort_values(by='id lead').reset_index(drop=True)
leads_credito_activos['diff_leads'] = leads_credito_activos['id lead'].diff().shift(-1).fillna(0).astype(int)
leads_credito_activos

,espacio automarket,id lead,diff_leads
0,MetrÃ³poli Patriotismo,16,50
1,Samara SatÃ©lite,66,32
2,Reforma 510,98,0


In [ ]:
# df de asignación para emparejar los leads activos en espacios
asign_espacio_justiciera = leads_credito_activos.loc[
    leads_credito_activos.index.repeat(leads_credito_activos["diff_leads"])
]['espacio automarket'].reset_index(drop=True) # sale como una serie
asign_espacio_justiciera = asign_espacio_justiciera.to_frame()

# df de asignación normal
asign_espacio_normal = leads_credito_activos.loc[
    leads_credito_activos.index
]['espacio automarket'].reset_index(drop=True).to_frame()

# Ahora vamos a pegar tantas veces como sea necesario la asignación normal
lte = len(leads_nvos) - len(asign_espacio_justiciera) # leads tras emparejamiento
repeticiones_carrusel = ( math.ceil(lte/len(asign_espacio_normal)) ) if lte>0 else 0
print(f'Después de emparejar los leads de crédito en cada espacio, vamos a pasarlos {repeticiones_carrusel} veces por el carrusel')

asign_espacio = asign_espacio_justiciera.copy()
for i in range(repeticiones_carrusel):
  asign_espacio = pd.concat([asign_espacio,asign_espacio_normal])
asign_espacio.reset_index(drop=True, inplace = True)
asign_espacio['llave_espacio'] = asign_espacio.index + 1
asign_espacio

Después de emparejar los leads de crédito en cada espacio, vamos a pasarlos 0 veces por el carrusel


,espacio automarket,llave_espacio
0,MetrÃ³poli Patriotismo,1
1,MetrÃ³poli Patriotismo,2
2,MetrÃ³poli Patriotismo,3
3,MetrÃ³poli Patriotismo,4
4,MetrÃ³poli Patriotismo,5
...,...,...
77,Samara SatÃ©lite,78
78,Samara SatÃ©lite,79
79,Samara SatÃ©lite,80
80,Samara SatÃ©lite,81


In [ ]:
leads_nvos = leads_nvos.reset_index(drop=True)
leads_nvos['llave_espacio'] = leads_nvos.index + 1
leads_nvos = leads_nvos.merge(asign_espacio, how='left', on = 'llave_espacio')
leads_nvos['espacio automarket'] = leads_nvos['espacio automarket'].replace(dicc_espacios)

In [ ]:
leads_nvos

,id_comprador,phone,email,nb_comprador,tp_solicitud,folio,llave_espacio,espacio automarket
0,x,9994150476,x,KIMBERLY AREYSI MIGUEL ARGUETA,EDA Crédito,9733695,1,patriotismo
1,x,5621830204,x,LUIS REBOLLO TINOCO,EDA Crédito,9720835,2,patriotismo
2,x,5628470884,x,JUAN CRUZ SANCHEZ,EDA Crédito,9724896,3,patriotismo
3,x,5571295559,x,JOSE ANTONIO RODRIGUEZ SANTIAGO,EDA Crédito,9724817,4,patriotismo
4,x,5649965974,x,BLANCA ESTELA BAUTISTA MEJIA,EDA Crédito,9732002,5,patriotismo
5,x,7121598144,x,SAMUEL HERMENEGILDO SANCHEZ,EDA Crédito,9721172,6,patriotismo
6,x,7291650758,x,JESUS HERNANDEZ CRUZ,EDA Crédito,9722596,7,patriotismo
7,x,5573381915,x,SWEETY MELISSA CORONA MADRIGAL,EDA Crédito,9723697,8,patriotismo
8,x,5611837772,x,AARON AMAURY SANTANA SEGURA,EDA Crédito,9554713,9,patriotismo
9,x,5531409786,x,AGRIPINO ESPINOZA VAZQUEZ,EDA Crédito,9581098,10,patriotismo


### Asignacion asesores

In [ ]:
# Leemos catálogo de asesores
cat_as = list_file_ids_for_drive_folder(drive, id_drive_catalogos)['AsesoresEspacio']
centros = ['torre','samara','patriotismo','celula_credito']
assrs_actvs = {}
for c in centros:
  df = read_from_google_sheets(gc,cat_as,c)
  df['espacio'] = c
  df = df[df.activo==1].drop_duplicates(['asesor','espacio'])
  assrs_actvs[c] = df
assrs_actvs = pd.concat(assrs_actvs.values())
assrs_actvs = assrs_actvs.reset_index(drop=True)
assrs_actvs['asesor'] = assrs_actvs['asesor'].str.lower().str.strip()

# Número de asesores activos por espacio
num_assrs = {c: len(assrs_actvs[assrs_actvs.espacio==c]) for c in assrs_actvs.espacio.unique()}
num_assrs

In [ ]:
centros_apagados = [c for c in centros if len(assrs_actvs[assrs_actvs.espacio==c])==0]
centros_activos = [c for c in centros if c not in centros_apagados]
print(f'Espacio(s) apagado(s): {centros_apagados}. Se redistribuyen sus pedidos en los otros espacios activos.')

leap = leads_nvos['espacio'].isin(centros_apagados) # Leads en Espacio Apagado
n = leap.sum()
leads_nvos.loc[leap, 'espacio'] = np.random.choice(centros_activos, size = n)

In [ ]:
# Leads activos por asesor

leads_asesor_espacio = hist[~hist['estatus de lead'].isin(['COMPRA EXITOSA ','COMPRA EXITOSA', 'CERRADO'])].groupby(['asesor espacio','espacio automarket']).agg({'id lead':'nunique'}).reset_index()
leads_asesor_espacio.columns = ['asesor','espacio','leads']
leads_asesor_espacio['asesor'] = leads_asesor_espacio['asesor'].str.lower().str.strip()
leads_asesor_espacio['espacio'] = leads_asesor_espacio['espacio'].replace(dicc_espacios)
leads_asesor_espacio = leads_asesor_espacio[~leads_asesor_espacio['asesor'].str.strip().isin(['PRUEBA','prueba','#N/A()','#n/a ()'])]
leads_asesor_espacio = leads_asesor_espacio.groupby(['asesor','espacio']).agg({'leads':'sum'}).reset_index()

leads_asesor_cred = hist[~hist['estatus de lead'].isin(['COMPRA EXITOSA ','COMPRA EXITOSA', 'CERRADO'])].groupby(['asesor credito']).agg({'id lead':'nunique'}).reset_index()
leads_asesor_cred.columns = ['asesor','leads']
leads_asesor_cred['asesor'] = leads_asesor_cred['asesor'].str.lower().str.strip()
leads_asesor_cred['espacio'] = 'celula_credito'
leads_asesor_cred = leads_asesor_cred[~leads_asesor_cred['asesor'].str.strip().isin(['PRUEBA', 'prueba', '#N/A()', '#n/a ()', '', ' '])]
leads_asesor_cred = leads_asesor_cred.groupby(['asesor','espacio']).agg({'leads':'sum'}).reset_index()

leads_asesor = pd.concat([leads_asesor_espacio, leads_asesor_cred])

assrs_actvs_leads = assrs_actvs.merge(leads_asesor, how = 'left', on = ['asesor','espacio'])
assrs_actvs_leads = assrs_actvs_leads.sort_values(['espacio','leads']).reset_index(drop=True)

assrs_actvs_leads['llave'] = assrs_actvs_leads.groupby('espacio').cumcount()
assrs_actvs_leads['leads'] = assrs_actvs_leads['leads'].fillna(0)

leads_asesor_cred = assrs_actvs_leads[assrs_actvs_leads['espacio'] == 'celula_credito'].rename(columns = {'asesor':'asesor credito'})
leads_asesor_esp = assrs_actvs_leads[assrs_actvs_leads['espacio'] != 'celula_credito'].rename(columns = {'asesor':'asesor espacio'})
display(leads_asesor_cred)
display(leads_asesor_esp)

In [ ]:
leads_nvos = leads_nvos.rename(columns = {'espacio automarket':'espacio asig'}) #espacio asig es el espacio que asignamos en el paso previo
# Vamos a procesar distinto los leads que ya tuvieron gestión en algún espacio. Los seleccionamos con inner merge vs histórico

leads_nvos['phone'] = pd.to_numeric(leads_nvos['phone'],errors = 'coerce').astype('Int64')
hist['telefono comprador'] = pd.to_numeric(hist['telefono comprador'],errors = 'coerce').astype('Int64')
leads_nvos_comprprevio = leads_nvos.copy()
leads_nvos_comprprevio = leads_nvos_comprprevio.merge(
    hist[['telefono comprador','espacio automarket','asesor espacio']].rename(
    columns = {'espacio automarket':'espacio previo', 'asesor espacio':'asesor espacio previo'}
    ).drop_duplicates(
        'telefono comprador',keep='first'),
                                        how = 'inner', left_on = 'phone', right_on = 'telefono comprador').drop(columns = ['telefono comprador'])

leads_nvos_comprprevio['espacio previo'] = leads_nvos_comprprevio['espacio previo'].replace(dicc_espacios)

# Damos por bueno el espacio que tenía en su lead anterior; mergeamos por espacio previo con catálogo de asesores y nos quedamos con el primer asesor que cruce y no haya tenido antes (columna aux)
leads_nvos_comprprevio = leads_nvos_comprprevio.merge(leads_asesor_esp.rename(columns = {'asesor espacio':'asesor espacio nvo'}), how = 'left', left_on = 'espacio previo', right_on = 'espacio')
leads_nvos_comprprevio['aux'] = (leads_nvos_comprprevio['asesor espacio previo'].str.lower().str.strip() == leads_nvos_comprprevio['asesor espacio nvo'].str.lower().str.strip())*1
leads_nvos_comprprevio = leads_nvos_comprprevio.sort_values(by='aux', ascending=True)
leads_nvos_comprprevio = leads_nvos_comprprevio.drop_duplicates(['phone'], keep = 'first').reset_index(drop=True)
leads_nvos_comprprevio = leads_nvos_comprprevio.drop(columns = ['espacio asig','espacio previo','asesor espacio previo','activo','leads','llave','aux']).rename(columns = {'asesor espacio nvo':'asesor espacio'})
display(leads_nvos_comprprevio)

In [ ]:
# Aqui procesamos los leads nuevos con comprador nuevo
leads_nvos_comprnvo = leads_nvos[~leads_nvos['phone'].isin(leads_nvos_comprprevio['phone'].unique())].copy()
leads_nvos_comprnvo['llave_esp'] = leads_nvos_comprnvo.groupby('espacio asig').cumcount() % leads_nvos_comprnvo['espacio asig'].map(num_assrs)
leads_nvos_comprnvo = leads_nvos_comprnvo.merge(leads_asesor_esp, how='left', left_on = ['espacio asig','llave_esp'], right_on = ['espacio','llave'])
leads_nvos_comprnvo

In [ ]:
# Juntamos leads de compradores previos y nuevos, y asignamos asesor de credito
salida_leads = pd.concat([leads_nvos_comprnvo, leads_nvos_comprprevio]).sort_values(by='id_comprador').reset_index(drop=True)
salida_leads['llave_celcred'] = salida_leads.index % num_assrs['celula_credito']
salida_leads = salida_leads.merge(leads_asesor_cred, how = 'left', left_on = 'llave_celcred', right_on = 'llave', suffixes = ['','_cred'])
salida_leads

## Formato de salidas

In [ ]:
# Leemos catálogo de nomenclatura de leads
cat_nomLeads = list_file_ids_for_drive_folder(drive, id_drive_catalogos)['NomenclaturaLeads']
cat_nomLeads = read_from_google_sheets(gc, cat_nomLeads)
cat_nomLeads = cat_nomLeads[['Tipo de Lead','Clave']]

In [ ]:
salida_leads = salida_leads.reset_index(drop=True)
salida_leads['index'] = salida_leads.index + 1
salida_leads['id lead'] = ultimo_lead + salida_leads['index']
salida_leads = salida_leads.merge(cat_nomLeads, how = 'left', left_on = 'tp_solicitud', right_on = 'Tipo de Lead')
# Validamos que todos tengan nomenclatura asignada
if (salida_leads['Clave'].isna().sum() > 0):
  print(f'{color.RED} Algo falló en la nomenclatura de leads. Hay algunos sin clave{color.END}')
  raise SystemExit

salida_leads['id lead'] = salida_leads['Clave'] + '-' + salida_leads['id lead'].astype(str).str.zfill(6)
salida_leads['folio'] = salida_leads['folio'].fillna('x')
salida_leads.drop(columns = ['llave','index','activo','leads'], inplace = True)
salida_leads['cosecha'] = cosecha
salida_leads['fecha de asignacion'] = fh_de_asignacion.replace('-','/')

salida_leads.rename(columns = {'id_comprador':'id comprador','espacio':'espacio automarket','phone':'telefono comprador','email':'mail comprador',
                               'nb_comprador':'nombre comprador','tp_solicitud':'origen automarket','folio':'folio bauto tc'},inplace=True)
salida_leads = salida_leads[['id lead','origen automarket','cosecha','id comprador','folio bauto tc',
                             'nombre comprador','mail comprador','telefono comprador','asesor credito','espacio automarket','asesor espacio','fecha de asignacion']]

salida_leads['espacio automarket'] = salida_leads['espacio automarket'].replace(dicc_espacios3)
salida_leads['asesor credito'] = salida_leads['asesor credito'].str.title()
salida_leads['asesor espacio'] = salida_leads['asesor espacio'].str.title()
salida_leads['estatus de lead'] = 'celula de credito'

salida_leads

In [ ]:
# Leads activos por asesor iniciales
hca = hist_credito_activos[['espacio automarket','asesor espacio','asesor credito','origen automarket','id lead']]
hca['espacio automarket'] = hca['espacio automarket'].replace(dicc_espacios)
hca['origen automarket'] = hca['origen automarket'].str.strip().str.split(' ').str[0].replace('Apificado','apis').replace('Contingencia','contingencia') # Nos quedamos con la primera palabra, ya sea apificado o contingencia
hca = hca[~hca['asesor espacio'].str.strip().isin(['PRUEBA','prueba','#N/A()','#n/a ()'])]

hca_ases = hca.copy().rename(columns = {'id lead':'leads','asesor espacio':'asesor','espacio automarket':'espacio'})
hca_ases = hca_ases.pivot_table(index=['espacio', 'asesor'], columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
hca_ases.columns = hca_ases.columns.map(lambda col: '_'.join([str(x) for x in col if x]))

hca_ascr = hca.copy().rename(columns = {'id lead':'leads','asesor credito':'asesor'})
hca_ascr = hca_ascr.pivot_table(index=['asesor'], columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
hca_ascr.columns = hca_ascr.columns.map(lambda col: '_'.join([str(x) for x in col if x]))
hca_ascr['espacio'] = 'celula_credito'

hca = pd.concat([hca_ases, hca_ascr])
hca['asesor'] = hca['asesor'].str.lower().str.strip()
assrs_actvs_leads = assrs_actvs.merge(hca, how = 'left', on = ['asesor','espacio'])
assrs_actvs_leads = assrs_actvs_leads[['espacio','asesor','activo','leads_apis','leads_contingencia']].fillna(0)
assrs_actvs_leads['espacio'] = assrs_actvs_leads['espacio'].replace(dicc_espacios3)

# Leads asignados por asesor

res_asign = salida_leads.copy()
res_asign['asesor espacio'] = res_asign['asesor espacio'].str.lower().str.strip()
res_asign['asesor credito'] = res_asign['asesor credito'].str.lower().str.strip()
res_asign['origen automarket'] = res_asign['origen automarket'].str.strip().str.split(' ').str[0].replace('Apificado','apis').replace('Contingencia','contingencia')

asgns_ases = res_asign.copy().rename(columns = {'espacio automarket':'espacio','asesor espacio':'asesor','id lead':'leads'})
asgns_ases = asgns_ases.pivot_table(index=['espacio','asesor'],columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
asgns_ases.columns = asgns_ases.columns.map(lambda col: '_'.join([str(x) for x in col if x]))

asgns_ascr = res_asign.copy().rename(columns = {'espacio automarket':'espacio','asesor credito':'asesor','id lead':'leads'})
asgns_ascr = asgns_ascr.pivot_table(index=['asesor'], columns = 'origen automarket', aggfunc = {'leads':'nunique'}).reset_index()
asgns_ascr.columns = asgns_ascr.columns.map(lambda col: '_'.join([str(x) for x in col if x]))
asgns_ascr['espacio'] = 'celula_credito'

res_asign = pd.concat([asgns_ases, asgns_ascr]).fillna(0)

res_asign = assrs_actvs_leads.merge(res_asign, how = 'left', on = ['espacio','asesor'], suffixes = ['_iniciales','_nuevos'])
res_asign.fillna(0,inplace = True)
try:
  res_asign['leads_apis_finales'] = res_asign['leads_apis_iniciales'] + res_asign['leads_apis_nuevos']
except:
  print(f'No hay asignación nueva de apis')
try:
  res_asign['leads_contingencia_finales'] = res_asign['leads_contingencia_iniciales'] + res_asign['leads_contingencia_nuevos']
except:
    print(f'No hay asignación nueva de contingencia')

res_asign

In [ ]:
cols_agg = [c for c in res_asign.columns if all(x != c for x in ['espacio','asesor','activo']) ]
sums = res_asign.groupby('espacio').agg({c: 'sum' for c in cols_agg}).reset_index()
stds = res_asign.groupby('espacio').agg({c: 'std' for c in cols_agg}).reset_index()
sums['asesor'] = 'sum'
stds['asesor'] = 'std'
sums['activo'] = 0
stds['activo'] = 0

res_asign = pd.concat([res_asign,sums,stds]).reset_index(drop=True)
res_asign[cols_agg] = res_asign[cols_agg].round(1)
res_asign

In [ ]:
# Celda de validacion tipo assert
id_l_nvos = salida_leads['id lead'].unique()

if not hist[hist['id lead'].isin(id_l_nvos)].shape[0] == 0:
    print(f"{color.RED}Se están duplicando id leads respecto a los que ya existían en la torre de control{color.END}")
    raise SystemExit

## Guardado

In [ ]:
# Creamos la hoja de sheets de cero

ahora_dt = datetime.now(ZoneInfo("America/Mexico_City")).strftime('%d-%m-%Y %H:%M')
nb_sheets_salida = f'Salida {ahora_dt}'
nb_hojas = ['asignacion_edas','resumen_asignaciones']

id_folder_mes_salida = list_file_ids_for_drive_folder(drive, id_drive_salidas)[nb_carpeta_ctes_mes]
create_sheets_in_drive_folder(gc, nb_sheets_salida, id_folder_mes_salida)
id_sheets_salida = listar_archivos(id_folder_mes_salida, mime_types='application/vnd.google-apps.spreadsheet')[nb_sheets_salida]

# Creamos hojas y eliminamos la hoja 1.
# Si esto da error, muy probablemente es porque ya hay un archivo con el mismo nombre en la carpeta. -----> Revisa la carpeta de drive.
crear_hojas_sheets(id_sheets_salida, nb_hojas)
borrar_hojas(id_sheets_salida,['Hoja 1'])

In [ ]:
# Guardamos la salida del proceso de asignación en hojas de respaldo

hoja_df = {'asignacion_edas': salida_leads, 'resumen_asignaciones':res_asign}
for hoja, df in hoja_df.items():
  update_sheets_in_drive_folder(gc, id_sheets_salida, hoja, df)
  formato_hojas_sheets(id_sheets_salida, [hoja], n_columnas = df.shape[1], letra = 'Source Serif 4' )

print(f'Las salidas se generaron y se guardaron en https://docs.google.com/spreadsheets/d/{id_sheets_salida}')

In [ ]:
# Guardamos el corte con el que trabajamos en esta asignación

# primero el full de los edas con cortede ejercicio previo, después el full con corte de este ejercicio
corte = pd.concat([edas_ultimoCorte, edas_full])
# fh_corte cambia en los cortes, por lo que la excluimos del criterio de duplicidad
corte['Folio Preautorizado'] = pd.to_numeric(corte['Folio Preautorizado'],errors='coerce').astype('Int64')
corte = corte.drop_duplicates('Folio Preautorizado',
                              # priorizamos los que ya estaban en el último corte (por su etiqueta de fh_corte)
                              keep='first')
write_csv_to_drive(drive, insumos_edas[nb_ultimo_corte], corte)

In [ ]:
# Tomamos foto a torre de control
ahora_dt = datetime.now(ZoneInfo("America/Mexico_City")).strftime('%d-%m-%Y %H:%M')
nb_foto = f'FotoTC_{ahora_dt}'
tc2_foto = read_from_google_sheets(gc, id_sheets_tc2, sheetname='asignacion')

# guardamos respaldo
crear_hojas_sheets(id_sheets_salida, [nb_foto])
update_sheets_in_drive_folder(gc, id_sheets_salida, nb_foto, tc2_foto)
formato_hojas_sheets(id_sheets_salida, [nb_foto], n_columnas = tc2_foto.shape[1], letra = 'Source Serif 4' )
# actualizamos si aplica
if actualizar_tc == 'S':
  append_dataframe_to_google_sheet_from_range(gc, id_sheets_tc2, 'asignacion', salida_leads)
else:
  print('No se actualizó la torre de control')

In [ ]:
# # Guardamos el primer corte de edas.
# edas_ultimoCorte = edas_full.copy().reset_index(drop=True)
# edas_ultimoCorte['fh_corte'] = '24-02-2026 18'
# edas_ultimoCorte = edas_ultimoCorte[:-14]
# edas_ultimoCorte
# create_csv_file_in_drive_folder(drive, id_drive_edas, edas_ultimoCorte, nb_ultimo_corte)